<a href="https://colab.research.google.com/github/MO7AMEDNABIL/AI-Powered-Assistant-for-Churn-Prediction/blob/main/LLM_Pipeline_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# load the ML model


In [ ]:
import joblib
import shap

xgb_model = joblib.load("xgb_churn_model.pkl")
feature_columns = joblib.load("feature_columns.pkl")
explainer = shap.TreeExplainer(xgb_model)

# handel unneeded values

In [ ]:
def prepare_customer(customer):
    features = {col: 0 for col in feature_columns}

    # 1. Extract raw values safely
    raw_tenure = customer.get("tenure")
    raw_monthly = customer.get("Monthly_Charges")
    raw_contract = customer.get("Contract")

    # 2. Enforce absolute fallbacks/types locally FIRST
    clean_tenure = int(raw_tenure) if raw_tenure is not None else 1
    clean_monthly = float(raw_monthly) if raw_monthly is not None else 65.0
    clean_contract = int(raw_contract) if raw_contract is not None else 0

    # 3. Assign sanitized values to the features dictionary
    features["tenure"] = clean_tenure
    features["Monthly_Charges"] = clean_monthly
    features["Contract"] = clean_contract

    # 4. Perform math safely with verified numbers
    features["Total_Charges"] = clean_tenure * clean_monthly

    features["Is_Married"] = 1 if customer.get("Is_Married") in [True, 1, "1"] else 0
    features["Dependents"] = 1 if customer.get("Dependents") in [True, 1, "1"] else 0
    features["Senior_Citizen "] = 1 if customer.get("Senior_Citizen") in [True, 1, "1"] else 0

    # 6 set baseline assumptions
    features["Phone_Service"] = 1
    features["Paperless_Billing"] = 1

    # 7 Handle categoricals
    internet = customer.get("Internet_Service", "")
    if internet == "Fiber optic":
        features["Internet_Service_Fiber optic"] = 1
    elif internet == "DSL":
        features["Internet_Service_DSL"] = 1

    payment = customer.get("Payment_Method", "")
    if payment == "Electronic check":
        features["Payment_Method_Electronic check"] = 1
    elif payment == "Credit card":
        features["Payment_Method_Credit card (automatic)"] = 1
    elif payment == "Bank transfer":
        features["Payment_Method_Bank transfer (automatic)"] = 1

    return pd.DataFrame([features])

# Install the LLM library

In [ ]:
pip install unsloth

# Take user input and covert to JSON

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer
import torch
import json
import re

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

# System Prompt
system_prompt = """
You are a telecom churn feature extractor.

Extract customer information and return ONLY valid JSON.

Schema:

{
  "tenure": int,
  "Contract": int,
  "Internet_Service": string,
  "Payment_Method": string,
  "Monthly_Charges": float,
  "Is_Married": bool,
  "Dependents": bool,
  "Senior_Citizen": bool
}

Normalization Rules:

Contract:
- If the user explicitly mentions "month-to-month", use 0.
- If the user explicitly mentions "one year", use  "1".
- If the user explicitly mentions "two year" or "two years", use"2".
- DO NOT infer "One year" or "Two year" from tenure.
- If contract is not mentioned and tenure is expressed in months (e.g. "5 months", "12 months"), assume "Month-to-month".
- Otherwise use null.

Internet_Service:
- "fiber" -> "Fiber optic"
- "fiber optic" -> "Fiber optic"
- "dsl" -> "DSL"
- "no internet" -> "No"

Payment_Method:
- "electronic check" -> "Electronic check"
- "credit card" -> "Credit card"
- "bank transfer" -> "Bank transfer"
- "mailed check" -> "Mailed check"

Marital Status:
- married -> 1
- not married, single, divorced -> 0

Dependents:
- has dependents -> 1
- no dependents -> 0

Senior_Citizen:
- Set to true ONLY if the user explicitly mentions:
  "senior citizen", "elderly", "retired", "pensioner", or age 65+.
- If none of these are explicitly mentioned, set to "false".
- Never infer senior citizen status from tenure, marital status, dependents, or any other information.

Tenure:
- Extract the numeric number of months.
- "5 months" -> 5
- "2 years" -> 24

Monthly_Charges:
- Extract the monthly charge amount as a float.

Rules:
- Return JSON only.
- No explanation.
- No markdown.
- Use null when information is missing.
"""

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
pip install fastapi uvicorn pyngrok

# Genarate the report

In [ ]:
import os
import uvicorn
import threading
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import pandas as pd

nest_asyncio.apply()

STRATEGY_MAP = {
    "Stable Loyalty Tier": {
        "focus": "Focus entirely on deep loyalty, increasing lifetime value, and subtle upsell opportunities.",
        "analysis": "The account is securely in the Stable Loyalty Tier with an exact {prob}% risk. Stability is anchored by long-term indicators.",
        "recommendations": [
            "Leverage account stability to introduce premium tenure rewards.",
            "Offer subtle value-added digital service bundles to increase lifetime value.",
            "Engage the user with proactive account health check-ins."
        ],
        "conclusion": "The customer represents a highly stable account with strong retention indicators."
    },
    "Active Transition Monitoring Tier": {
        "focus": "Focus on active customer success check-ins, stabilization, and identifying friction points.",
        "analysis": "The account is trending into the Active Transition Monitoring Tier with a moderate {prob}% risk, showing initial signs of vulnerability.",
        "recommendations": [
            "Schedule a proactive customer success check-in to evaluate service satisfaction.",
            "Review billing patterns to ensure the customer is on the most optimized tier.",
            "Provide targeted educational content showcasing underutilized account features."
        ],
        "conclusion": "The account requires active monitoring to stabilize engagement and mitigate rising risks."
    },
    "Critical Save Tier": {
        "focus": "Focus on immediate churn prevention, aggressive retention saves, and high-priority interventions.",
        "analysis": "The account is flagged in the Critical Save Tier with a high {prob}% churn risk. Immediate defensive intervention is required.",
        "recommendations": [
            "Deploy an immediate contract-save incentive or a high-value discount offer.",
            "Route this account directly to the senior retention specialist team for a direct save call.",
            "Address the key cost and contract friction points immediately to secure a renewal."
        ],
        "conclusion": "This account is at imminent risk of loss; aggressive retention actions must be executed immediately."
    }
}


def generate_churn_report(prob: float, top_factors: list) -> str:
    if prob < 30:
        tier = "Stable Loyalty Tier"
    elif prob <= 60:
        tier = "Active Transition Monitoring Tier"
    else:
        tier = "Critical Save Tier"

    strategy = STRATEGY_MAP[tier]
    factors_text = "\n".join(f"- {factor}" for factor in top_factors)
    recommendations_text = "\n".join(f"- {rec}" for rec in strategy["recommendations"])

    return f"""## Prediction Analysis
{strategy['analysis'].format(prob=prob)}

## Key Risk Factors
{factors_text}

## Strategic Business Recommendations
{recommendations_text}

## Conclusion
{strategy['conclusion']}
"""


def remove_code_fences(text: str) -> str:
    return re.sub(r"```[\w]*\n?|```", "", text).strip()

app = FastAPI(title="ChurnGuard AI")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class CustomerInput(BaseModel):
    prompt: str


@app.post("/predict")
async def predict(customer: CustomerInput):
    try:
        user_prompt = customer.prompt

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=False)

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response.split("assistant")[-1].strip()
        cleaned = remove_code_fences(response)
        customer_data = json.loads(cleaned)

        customer_df = prepare_customer(customer_data)
        probability = xgb_model.predict_proba(customer_df)[0][1]
        prediction = "Churn" if probability >= 0.5 else "No Churn"
        shap_values = explainer(customer_df)

        if len(shap_values.values.shape) == 3:
            local_shap = shap_values.values[0, :, 1]
        else:
            local_shap = shap_values.values[0]

        feature_impacts = dict(zip(feature_columns, local_shap))
        sorted_factors = sorted(feature_impacts.items(), key=lambda x: abs(x[1]), reverse=True)

        top_factors = []
        for feat, val in sorted_factors[:3]:
            direction = "increases risk" if val > 0 else "reduces risk"

            if customer_df[feat].iloc[0] == 0:
                if feat == "Contract":
                    top_factors.append(f"Month-to-month plan ({direction})")
                elif "Payment_Method" in feat and val < 0:
                    top_factors.append(f"Not using {feat.split('_')[-1]} ({direction})")
                elif "Internet_Service" in feat and val < 0:
                    top_factors.append(f"Not using {feat.split('_')[-1]} ({direction})")
                else:
                    clean_feat_name = feat.replace("_", " ").title()
                    top_factors.append(f"Absence of {clean_feat_name} ({direction})")
            else:
                if feat == "tenure":
                    top_factors.append(f"Customer Tenure ({direction})")
                elif feat == "Total_Charges":
                    top_factors.append(f"High Lifetime Spend ({direction})")
                elif feat == "Monthly_Charges":
                    top_factors.append(f"Monthly Subscription Fee ({direction})")
                else:
                    clean_feat_name = feat.replace("_", " ").title()
                    top_factors.append(f"{clean_feat_name} ({direction})")

        prob_percentage = round(float(probability) * 100, 1)

        report = generate_churn_report(
            prob=prob_percentage,
            top_factors=top_factors,
        )

        return {
            "prediction": prediction,
            "probability": prob_percentage,
            "top_factors": top_factors,
            "report": report,
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/health")
async def health():
    return {"status": "ok"}


def start_server():
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()


if __name__ == "__main__":
    try:
        ngrok.kill()
    except Exception:
        pass

    ngrok.set_auth_token("31DU5Da2L0HY0cmcLbl26EbTXgd_VrWEdKc4gKazMiZokwzL")

    tunnel = ngrok.connect(8000, "http")
    public_url = tunnel.public_url

    print("\n" + "=" * 60)
    print("  ChurnGuard AI — Backend Running")
    print("=" * 60)
    print(f"  Local  : http://localhost:8000")
    print(f"  Public : {public_url}")
    print("=" * 60 + "\n")

    server_thread = threading.Thread(target=start_server, daemon=True)
    server_thread.start()
    import time
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nStopping background server...")
    finally:
        ngrok.kill()

INFO:     Started server process [7052]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



  ChurnGuard AI — Backend Running
  Local  : http://localhost:8000
  Public : https://be19-34-158-12-125.ngrok-free.app

INFO:     197.51.50.251:0 - "OPTIONS /predict HTTP/1.1" 200 OK


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

INFO:     197.51.50.251:0 - "POST /predict HTTP/1.1" 200 OK


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


INFO:     197.51.50.251:0 - "POST /predict HTTP/1.1" 200 OK


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


INFO:     197.51.50.251:0 - "POST /predict HTTP/1.1" 200 OK

Stopping background server...
